In [8]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import load_workbook

In [9]:
folder = Path("../sqllab_fkp/")
# files = list(folder.glob("sqllab_assignment_for_fkp*.xlsx"))
files = list(folder.glob("*.csv"))

if not files:
    raise FileNotFoundError("Tidak ada file Excel di folder sqllab_fkp")


print(f"Ditemukan {len(files)} file:")
for f in files:
    print(" -", f.name)

# Baca semua file lalu gabungkan
list_df = []
for f in files:
    df = pd.read_csv(f, dtype=str)
    df["source_file"] = (
        f.name
    )  # opsional: kolom penanda asal file, bisa dihapus kalau tidak perlu
    list_df.append(df)

df_gabungan = pd.concat(list_df, ignore_index=True)

print(f"\nTotal baris setelah digabung: {len(df_gabungan)}")
df_gabungan.head()

Ditemukan 1 file:
 - hasil_scraping_20260914_112112.csv

Total baris setelah digabung: 38340


,assignment_id,assignment_status_alias,level_6_full_code,no_bang,alamat_prelist,nama_kk_or_usaha,keberadaan_usaha,keberadaan_kk,jumlah_usaha,jumlah_usaha_ditemukan,jumlah_ak,index1,nama_usaha,keberadaan_usaha_label,kategori,kbli_label,source_file
0,ad76bd3b-0fa1-408f-b9ac-d89b3045faf2,APPROVED BY Pengawas,1671011001000400,NaN,JL. TALANG JAWA,ADANG KURNIAWAN,NaN,0. Tidak Ditemukan (STOP),0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,hasil_scraping_20260914_112112.csv
1,01e0f362-b6a8-4bac-b703-d19a180c2592,APPROVED BY Pengawas,1671011001000400,NaN,JL. TALANG JAWA,ANDI SUSANTO,NaN,0. Tidak Ditemukan (STOP),0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,hasil_scraping_20260914_112112.csv
2,2a3eea91-cb42-457f-8f76-f889998fce13,APPROVED BY Pengawas,1671011001000400,NaN,JL. PRINTIS TALANG JAWA,ANDRE LIANSYAH,NaN,0. Tidak Ditemukan (STOP),0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,hasil_scraping_20260914_112112.csv
3,c1a9e410-cfa2-4e1f-b0d3-950f66d5193f,APPROVED BY Pengawas,1671011001000400,NaN,JL TALANG JAWA,ARI USMAN,NaN,0. Tidak Ditemukan (STOP),0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,hasil_scraping_20260914_112112.csv
4,998f10ee-f0c9-4de6-b249-90ab43aba31a,APPROVED BY Pengawas,1671011001000400,NaN,"JL SOSIAL PT ABP 014 005, RT 0/RW 0",BENGKEL YANI,0. Tidak Ditemukan,NaN,1,0,NaN,1,BENGKEL YANI <ERNA>,0. Tidak Ditemukan,NaN,NaN,hasil_scraping_20260914_112112.csv


In [10]:
# df_gabungan = df_gabungan.drop(columns=["index1"])

df_gabungan["keberadaan"] = df_gabungan["keberadaan_usaha"].fillna(
    df_gabungan["keberadaan_kk"]
)

group_cols = [
    "assignment_id",
    "assignment_status_alias",
    "level_6_full_code",
    "alamat_prelist",
    "no_bang",
    "nama_kk_or_usaha",
    "keberadaan",
    "jumlah_usaha",
    "jumlah_usaha_ditemukan",
    "jumlah_ak",
]

concat_cols = ["nama_usaha", "keberadaan_usaha_label", "kategori", "kbli_label"]

def concat_unique(series, sep=" | "):
    vals = series.dropna().astype(str)
    vals = vals[vals.str.strip() != ""]
    return sep.join(vals.unique())


df_hasil = (
    df_gabungan.groupby(group_cols, dropna=False)
    .agg({col: concat_unique for col in concat_cols})
    .reset_index()
)

print(f"Total baris setelah groupby: {len(df_hasil)}")

Total baris setelah groupby: 37660


In [11]:
df_hasil.to_excel('../assigment_penyisiran_14092026.xlsx')